# 08 — MongoDB Serving Layer

**Airline Operations Intelligence Platform** · Notebook 8 of 10 · *runs locally*

## Purpose
Module 11 of the plan: push every precomputed mart into MongoDB, where the Streamlit
dashboard reads it. This is the **serving layer** — the boundary between the batch
analytics world (Spark, Parquet) and the interactive world (dashboard, sub-second reads).

## Syllabus coverage (Unit 3)
- **RDBMS vs NoSQL** — why a document store fits this access pattern
- **Data modelling in NoSQL** — denormalised, pre-aggregated, one document per dashboard question
- **Indexing** — measured with `explain()`, indexed vs unindexed
- **CAP theorem**, **sharding**, **replication** — documented in §7 and `docs/nosql_comparison.md`

## Setup
```bash
docker compose up -d          # starts mongo:7 on localhost:27017
```
Connection details come from `.env` (git-ignored), never hardcoded.

In [ ]:
import sys, os, time, json
sys.path.insert(0, "../src")

from config import build_spark, PATHS
from dotenv import load_dotenv
from pymongo import MongoClient, ASCENDING, DESCENDING

load_dotenv(PATHS["root"] / ".env")

MONGO_URI = os.getenv("MONGO_URI", "mongodb://localhost:27017")
MONGO_DB  = os.getenv("MONGO_DB", "airline_intel")

client = MongoClient(MONGO_URI, serverSelectionTimeoutMS=5000)
client.admin.command("ping")          # fails fast if the container is not running
db = client[MONGO_DB]

print(f"Connected to {MONGO_URI}")
print(f"Server version : {client.server_info()['version']}")
print(f"Database       : {MONGO_DB}")

---
## 1. Why a document database here (Unit 3)

The dashboard's access pattern is: *"give me every metric for airline X"* or
*"give me all airports for the map"*. Each answer is a **self-contained document** with no
joins required, because notebook 05 already did the joining and aggregation.

| Aspect | RDBMS | MongoDB | Fit for this project |
|---|---|---|---|
| Schema | Fixed, shared across rows | Per-document | Marts have different shapes — an airline document and a route document share no columns |
| Read pattern | Joins at query time | Denormalised single-document read | Dashboard reads one collection per page, no joins |
| Scaling | Vertical | Horizontal (sharding) | Not needed at 276 KB, but the model supports it |
| Format | Tables/rows | BSON documents | Maps directly to JSON for a web front-end |

**The honest counterpoint:** at this data size a relational database would work perfectly
well, and SQL would express the aggregations more naturally. MongoDB is the right choice
for the *serving* layer specifically — the shape of the data being served is document-like
and the read path needs no joins. The analytical work stays in Spark, where it belongs.

---
## 2. Load the marts

Spark reads the Parquet marts; pymongo writes them. The marts are small (hundreds to a few
thousand rows) so `toPandas()` is safe here — a full-dataset `toPandas()` would not be.

In [ ]:
spark = build_spark("08-mongodb")

# Discover marts rather than listing them: notebooks 06, 07 and 11 add collections
# over time, and a hardcoded list silently stops pushing whatever was added last.
mart_paths = sorted(p for p in PATHS["marts"].glob("*.parquet") if p.is_dir())

frames = {}
for path in mart_paths:
    frames[path.stem] = spark.read.parquet(str(path))

print(f"Discovered {len(frames)} marts\n")
print(f"{'MART':<28}{'ROWS':>8}{'COLS':>7}")
print("-" * 43)
for name, df in frames.items():
    print(f"{name:<28}{df.count():>8,}{len(df.columns):>7}")

---
## 3. Push to MongoDB

Each mart becomes one collection. The write is **idempotent** — the collection is dropped
and rebuilt, so re-running this notebook never duplicates documents.

NaN values are converted to `None`: BSON has no NaN, and leaving them produces documents
the dashboard cannot render.

In [ ]:
import math

def to_documents(df):
    """Spark DataFrame -> list of BSON-safe dicts."""
    records = df.toPandas().to_dict("records")
    for doc in records:
        for k, v in doc.items():
            if isinstance(v, float) and math.isnan(v):
                doc[k] = None
            elif hasattr(v, "item"):          # numpy scalar -> python scalar
                doc[k] = v.item()
    return records


t0 = time.time()
pushed = {}
for name, df in frames.items():
    docs = to_documents(df)
    db[name].drop()                            # idempotent rebuild
    if docs:
        db[name].insert_many(docs)
    pushed[name] = db[name].count_documents({})
    print(f"  {name:<28} {pushed[name]:>8,} documents")

print(f"\nPushed in {time.time()-t0:.1f}s")

In [ ]:
# Verify: MongoDB document counts must equal Spark row counts, per collection.
print(f"{'COLLECTION':<28}{'SPARK':>9}{'MONGO':>9}   OK")
print("-" * 52)
ok = True
for name, df in frames.items():
    spark_n, mongo_n = df.count(), pushed[name]
    match = spark_n == mongo_n
    ok &= match
    print(f"{name:<28}{spark_n:>9,}{mongo_n:>9,}   {'yes' if match else 'NO'}")
assert ok, "a collection does not match its mart"
print("\nEvery collection matches its source mart exactly.")

---
## 4. Document shape

Confirm the stored documents match the schemas in proposal §17.

In [ ]:
for name in ["overall_kpis", "airline_metrics", "airport_metrics", "route_metrics"]:
    doc = db[name].find_one({}, {"_id": 0})
    print(f"--- {name} ---")
    print(json.dumps(doc, indent=2, default=str)[:700])
    print()

---
## 5. Indexing

Without an index, MongoDB performs a **collection scan** — every document is examined.
An index turns that into a B-tree seek. The effect is measured below with `explain()`.

Indexes are chosen from the dashboard's actual query patterns, not speculatively.

In [ ]:
# Measure BEFORE indexing.
def scan_stats(coll, query):
    plan = db[coll].find(query).explain()
    ex = plan["executionStats"]
    stage = plan["queryPlanner"]["winningPlan"]
    # unwrap nested stages to find the access method
    s, kind = stage, None
    while s:
        kind = s.get("stage", kind)
        s = s.get("inputStage")
    return ex["totalDocsExamined"], ex["nReturned"], ex["executionTimeMillis"], kind

before = scan_stats("route_metrics", {"origin": "LAX", "destination": "SFO"})
print(f"BEFORE index -- docs examined: {before[0]:,}  returned: {before[1]}  "
      f"time: {before[2]}ms  stage: {before[3]}")

In [ ]:
db["airline_metrics"].create_index([("airline_code", ASCENDING)], unique=True)
db["airport_metrics"].create_index([("airport_code", ASCENDING)], unique=True)
db["airport_metrics"].create_index([("cluster_id", ASCENDING)])
db["route_metrics"].create_index([("origin", ASCENDING), ("destination", ASCENDING)])
db["route_metrics"].create_index([("delay_rate", ASCENDING)])
db["time_trends"].create_index([("dimension", ASCENDING), ("period", ASCENDING)])
db["airline_airport"].create_index([("airline_code", ASCENDING), ("airport_code", ASCENDING)])

print("Indexes created:\n")
for coll in ["airline_metrics", "airport_metrics", "route_metrics", "time_trends", "airline_airport"]:
    names = [ix["name"] for ix in db[coll].list_indexes()]
    print(f"  {coll:<22}{names}")

In [ ]:
after = scan_stats("route_metrics", {"origin": "LAX", "destination": "SFO"})
print(f"AFTER index  -- docs examined: {after[0]:,}  returned: {after[1]}  "
      f"time: {after[2]}ms  stage: {after[3]}")
print()
print(f"Documents examined fell from {before[0]:,} to {after[0]:,} "
      f"({before[0]/max(after[0],1):.0f}x fewer)")
print()
print("COLLSCAN -> IXSCAN is the observable difference. At 4,706 documents the wall-clock")
print("saving is small; the point is that the scan cost grows with collection size while")
print("the index seek does not. This is the same argument as partition pruning in Parquet.")

---
## 6. The queries the dashboard will actually run

Each dashboard page maps to one simple query. No joins, no aggregation at read time.

In [ ]:
queries = {
    "Overview KPI cards":        lambda: db.overall_kpis.find_one({}, {"_id": 0}),
    "Airline ranking table":     lambda: list(db.airline_metrics.find({}, {"_id": 0}).sort("delay_rate", 1)),
    "Airport map (clustered)":   lambda: list(db.airport_metrics.find({"cluster_id": {"$ne": None}}, {"_id": 0})),
    "Monthly trend":             lambda: list(db.time_trends.find({"dimension": "monthly"}, {"_id": 0}).sort("period", 1)),
    "Route lookup LAX->SFO":     lambda: db.route_metrics.find_one({"origin": "LAX", "destination": "SFO"}, {"_id": 0}),
    "Worst routes (min sample)": lambda: list(db.route_metrics.find({"meets_min_sample": True}, {"_id": 0}).sort("delay_rate", -1).limit(10)),
    "Delay causes":              lambda: list(db.delay_causes.find({}, {"_id": 0})),
}

print(f"{'DASHBOARD QUERY':<28}{'RESULT':>10}{'TIME':>10}")
print("-" * 48)
for label, fn in queries.items():
    t0 = time.time()
    out = fn()
    ms = (time.time() - t0) * 1000
    n = len(out) if isinstance(out, list) else (1 if out else 0)
    print(f"{label:<28}{n:>10,}{ms:>9.1f}ms")

In [ ]:
# The single most important read: everything the map page needs.
sample = db.airport_metrics.find_one({"cluster_id": {"$ne": None}}, {"_id": 0})
print("One airport document, exactly as the dashboard receives it:\n")
print(json.dumps({k: sample[k] for k in
      ["airport_code","airport_name","city","state","lat","lon","total_flights",
       "avg_delay","delay_rate","cancellation_rate","peak_delay_hour",
       "cluster_id","cluster_label"] if k in sample}, indent=2, default=str))

---
## 7. CAP, replication and sharding (Unit 3)

### CAP theorem applied to this deployment

MongoDB is a **CP** system: it favours Consistency and Partition tolerance over
Availability. In a replica set, all writes go to the primary and reads default to the
primary, so a client never sees a stale value. When the primary fails, the set holds an
election and **writes are unavailable for a few seconds** — availability is what gets
sacrificed.

For this workload that is the right trade: a dashboard showing stale operational metrics
is worse than one that pauses briefly.

**This deployment is a single node**, so CAP does not literally apply — there is no
partition to tolerate. The analysis describes how the system behaves when deployed as a
replica set, which is how MongoDB Atlas runs it.

### Replication
A production replica set is primary + 2 secondaries. Secondaries replicate the primary's
oplog and can serve reads if the application opts into eventual consistency
(`readPreference=secondaryPreferred`). This provides fault tolerance and read scaling.

### Sharding
Not required here — the entire serving layer is 276 KB. Documented for completeness:

| Collection | Sensible shard key | Why |
|---|---|---|
| `airport_metrics` | `airport_code` | High cardinality (322), evenly distributed, and every query filters on it |
| `route_metrics` | `{origin, destination}` | Compound key matching the dominant query |
| `airline_metrics` | *would not shard* | 14 documents; sharding adds overhead with no benefit |

A poor shard key — `cluster_id`, say, with only 4 values — would create hotspots by
concentrating writes on a few shards. Choosing a key is about cardinality and query
alignment, not just uniqueness.

---
## 8. Summary

In [ ]:
print(f"{'COLLECTION':<28}{'DOCS':>8}{'INDEXES':>10}")
print("-" * 46)
total = 0
for name in sorted(db.list_collection_names()):
    n = db[name].count_documents({})
    total += n
    print(f"{name:<28}{n:>8,}{len(list(db[name].list_indexes())):>10}")
print("-" * 46)
print(f"{'TOTAL':<28}{total:>8,}")

stats = db.command("dbStats")
print(f"\nDatabase size on disk : {stats['dataSize']/1024:.1f} KB")
print(f"Index size            : {stats['indexSize']/1024:.1f} KB")
print("\nThe dashboard now reads this instead of scanning 5.8 million rows.")

In [ ]:
client.close()
spark.stop()
print("Notebook 08 complete. Serving layer ready for the Streamlit dashboard.")